In [1]:
!pip install -q torch pandas scikit-learn datasets transformers accelerate

In [4]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(42)

In [5]:
train_df = pd.read_csv("PS_train.csv")
dev_df   = pd.read_csv("PS_dev.csv")
test_df  = pd.read_csv("PS_test_without_labels.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

Train shape: (4352, 2)
Dev shape: (544, 2)


In [6]:
label_names = sorted(train_df["labels"].unique())
print("Labels:", label_names)
print("Num labels:", len(label_names))

assert len(label_names) == 7, "Expected 7 classes"

label2id = {l:i for i,l in enumerate(label_names)}
id2label = {i:l for l,i in label2id.items()}

train_df["label_id"] = train_df["labels"].map(label2id)
dev_df["label_id"]   = dev_df["labels"].map(label2id)

Labels: ['Negative', 'Neutral', 'None of the above', 'Opinionated', 'Positive', 'Sarcastic', 'Substantiated']
Num labels: 7


In [7]:
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)

class_weights = torch.tensor(cw, dtype=torch.float32)
print("Class weights:", class_weights)
print("Weight shape:", class_weights.shape)

Class weights: tensor([1.5313, 0.9760, 3.6358, 0.4568, 1.0812, 0.7870, 1.5090])
Weight shape: torch.Size([7])


In [8]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
def tokenize(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = Dataset.from_pandas(
    train_df[["content","label_id"]]
    .rename(columns={"label_id":"labels"})
)

dev_ds = Dataset.from_pandas(
    dev_df[["content","label_id"]]
    .rename(columns={"label_id":"labels"})
)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["content"])
dev_ds   = dev_ds.map(tokenize, batched=True, remove_columns=["content"])

train_ds.set_format("torch")
dev_ds.set_format("torch")

Map:   0%|          | 0/4352 [00:00<?, ? examples/s]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [11]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    # training
    num_train_epochs=8,
    learning_rate=3e-5,
    weight_decay=0.01,

    # batch size
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    # logging
    logging_steps=50,
    report_to="none"
)


In [14]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
50,1.939488
100,1.786762
150,1.649565
200,1.654637
250,1.671517
300,1.604268
350,1.595340
400,1.590651
450,1.579610
500,1.550731


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1088, training_loss=1.5504533543306238, metrics={'train_runtime': 1842.3094, 'train_samples_per_second': 18.898, 'train_steps_per_second': 0.591, 'total_flos': 4580442872217600.0, 'train_loss': 1.5504533543306238, 'epoch': 8.0})

In [15]:
preds = trainer.predict(dev_ds)
y_pred = np.argmax(preds.predictions, axis=1)

print("\nClassification Report (DEV):")
print(classification_report(
    dev_df["label_id"],
    y_pred,
    target_names=label_names
))

macro = f1_score(dev_df["label_id"], y_pred, average="macro")
print("Macro F1:", macro)


Classification Report (DEV):
                   precision    recall  f1-score   support

         Negative       0.19      0.24      0.21        51
          Neutral       0.20      0.08      0.12        84
None of the above       1.00      0.90      0.95        20
      Opinionated       0.48      0.38      0.42       153
         Positive       0.24      0.46      0.32        69
        Sarcastic       0.48      0.42      0.45       115
    Substantiated       0.21      0.29      0.24        52

         accuracy                           0.35       544
        macro avg       0.40      0.40      0.39       544
     weighted avg       0.37      0.35      0.35       544

Macro F1: 0.38585794372755483


In [16]:
test_ds = Dataset.from_pandas(test_df[["content"]])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["content"])
test_ds.set_format("torch")

test_preds = trainer.predict(test_ds)
test_ids = np.argmax(test_preds.predictions, axis=1)

test_df["label"] = [id2label[i] for i in test_ids]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

In [17]:
test_df[["label"]].to_csv("Codeblitz_political.csv", index=False)
print("Submission file created: Codeblitz_political.csv")


Submission file created: Codeblitz_political.csv
